# Pandas 2 - Análisis Exploratorio de Datos (EDA)

## Introducción

En este notebook realizaremos un **análisis exploratorio completo** de la base de datos de películas IMDB. El análisis exploratorio de datos (EDA) es una etapa fundamental en cualquier proyecto de ciencia de datos que nos permite:

1. **Comprender la estructura** de los datos
2. **Identificar patrones** y relaciones entre variables
3. **Detectar valores atípicos** y datos faltantes
4. **Generar hipótesis** para análisis posteriores

El conjunto de datos contiene información de 1000 películas, incluyendo: ranking, título, género, descripción, director, actores, año, duración, calificación, votos, ingresos y metascore.

## 1. Importación de Librerías y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de gráficos
plt.rcParams.update({"font.size": 14, "figure.figsize": (12, 8)})

# Cargar datos
imdb_data = pd.read_csv("IMDB-Movie-Data.csv")

print("Datos cargados exitosamente.")
print(f"Dimensiones: {imdb_data.shape}")
print(f"Columnas: {list(imdb_data.columns)}")

## 2. Exploración Inicial de los Datos

### 2.1 Visualización de las primeras y últimas filas

In [ ]:
print("Primeras 5 filas:")
display(imdb_data.head())

print("\nÚltimas 5 filas:")
display(imdb_data.tail())

### 2.2 Información General del DataFrame

In [ ]:
print("Información del DataFrame:")
imdb_data.info()

In [ ]:
print(f"\nDimensiones: {imdb_data.shape[0]} filas × {imdb_data.shape[1]} columnas")
print(f"Tipos de datos:\n{imdb_data.dtypes.value_counts()}")

### 2.3 Estadísticas Descriptivas

In [ ]:
print("Estadísticas descriptivas:")
display(imdb_data.describe())

### 2.4 Análisis de Datos Faltantes

In [ ]:
missing_data = imdb_data.isnull().sum()
missing_percentage = (missing_data / len(imdb_data)) * 100

missing_df = pd.DataFrame({
    'Valores Faltantes': missing_data,
    'Porcentaje (%)': missing_percentage
})

print("Análisis de datos faltantes:")
display(missing_df[missing_df['Valores Faltantes'] > 0].sort_values('Valores Faltantes', ascending=False))

## 3. Análisis de Variables Numéricas

In [ ]:
numeric_cols = ['Rank', 'Year', 'Runtime (Minutes)', 'Rating', 'Votes', 'Revenue (Millions)', 'Metascore']

print("Resumen estadístico de variables numéricas:")
display(imdb_data[numeric_cols].describe())

### 3.1 Distribución de Calificaciones (Rating)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
imdb_data['Rating'].hist(bins=20, edgecolor='black', alpha=0.7)
plt.title('Distribución de Calificaciones')
plt.xlabel('Rating')
plt.ylabel('Frecuencia')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
imdb_data['Rating'].boxplot()
plt.title('Boxplot de Calificaciones')
plt.ylabel('Rating')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Estadísticas de Rating:")
print(f"  Media: {imdb_data['Rating'].mean():.2f}")
print(f"  Mediana: {imdb_data['Rating'].median():.2f}")
print(f"  Mínimo: {imdb_data['Rating'].min():.2f}")
print(f"  Máximo: {imdb_data['Rating'].max():.2f}")

### 3.2 Distribución de Ingresos (Revenue)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
imdb_data['Revenue (Millions)'].hist(bins=20, edgecolor='black', alpha=0.7)
plt.title('Distribución de Ingresos')
plt.xlabel('Revenue (Millions USD)')
plt.ylabel('Frecuencia')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
imdb_data['Revenue (Millions)'].boxplot()
plt.title('Boxplot de Ingresos')
plt.ylabel('Revenue (Millions USD)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Estadísticas de Ingresos:")
print(f"  Media: ${imdb_data['Revenue (Millions)'].mean():.2f}M")
print(f"  Mediana: ${imdb_data['Revenue (Millions)'].median():.2f}M")
print(f"  Mínimo: ${imdb_data['Revenue (Millions)'].min():.2f}M")
print(f"  Máximo: ${imdb_data['Revenue (Millions)'].max():.2f}M")

### 3.3 Relación Rating vs Revenue

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(imdb_data['Rating'], imdb_data['Revenue (Millions)'], alpha=0.5)
plt.xlabel('Rating')
plt.ylabel('Revenue (Millions USD)')
plt.title('Relación entre Calificación e Ingresos')
plt.grid(True, alpha=0.3)

# Línea de tendencia
z = np.polyfit(imdb_data['Rating'].dropna(), 
               imdb_data['Revenue (Millions)'].dropna(), 1)
p = np.poly1d(z)
plt.plot(imdb_data['Rating'].sort_values(), 
         p(imdb_data['Rating'].sort_values()), 
         'r--', label='Tendencia')

plt.legend()
plt.show()

correlation = imdb_data['Rating'].corr(imdb_data['Revenue (Millions)'])
print(f"Correlación entre Rating y Revenue: {correlation:.3f}")

### 3.4 Películas con Alto Rating y Bajo Revenue (Outliers)

In [ ]:
# Identificar películas con rating alto pero ingresos bajos
high_rating_low_revenue = imdb_data[
    (imdb_data['Rating'] > 8.0) & 
    (imdb_data['Revenue (Millions)'] < 50)
]

print(f"Películas con Rating > 8.0 y Revenue < $50M: {len(high_rating_low_revenue)}")
display(high_rating_low_revenue[['Title', 'Rating', 'Revenue (Millions)', 'Director', 'Year']].head(10))

## 4. Análisis de Variables Categóricas

In [ ]:
# Análisis de géneros
genres = imdb_data['Genre'].str.split(',').explode().str.strip()
genre_counts = genres.value_counts().head(10)

plt.figure(figsize=(12, 5))
genre_counts.plot(kind='bar')
plt.title('Top 10 Géneros de Películas')
plt.xlabel('Género')
plt.ylabel('Cantidad de Películas')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Distribución de Géneros:")
display(genre_counts)

In [ ]:
# Análisis de directores
director_counts = imdb_data['Director'].value_counts().head(10)

plt.figure(figsize=(12, 5))
director_counts.plot(kind='bar')
plt.title('Top 10 Directores con Más Películas')
plt.xlabel('Director')
plt.ylabel('Cantidad de Películas')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Directores con más películas:")
display(director_counts)

### 4.1 Análisis de Años de Publicación

In [ ]:
year_counts = imdb_data['Year'].value_counts().sort_index()

plt.figure(figsize=(14, 5))
year_counts.plot(kind='bar')
plt.title('Distribución de Películas por Año')
plt.xlabel('Año')
plt.ylabel('Cantidad de Películas')
plt.xticks(rotation=90)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Años con más películas:")
display(year_counts.head(5))

## 5. Análisis de Correlaciones

### 5.1 Matriz de Correlación

In [ ]:
# Seleccionar columnas numéricas para correlación
corr_cols = ['Rating', 'Votes', 'Revenue (Millions)', 'Metascore', 'Runtime (Minutes)']
corr_matrix = imdb_data[corr_cols].corr()

print("Matriz de Correlación:")
display(corr_matrix.style.background_gradient(cmap='coolwarm'))

# Visualización
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(np.arange(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(np.arange(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('Mapa de Calor - Correlaciones')

# Añadir valores
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        plt.text(i, j, f'{corr_matrix.iloc[i, j]:.2f}',
                 ha='center', va='center', color='white')
plt.tight_layout()
plt.show()

### 5.2 Correlaciones Destacadas

In [ ]:
print("Principales correlaciones con Rating:")
rating_corr = corr_matrix['Rating'].sort_values(ascending=False)
display(rating_corr)

print("\nPrincipales correlaciones con Revenue:")
revenue_corr = corr_matrix['Revenue (Millions)'].sort_values(ascending=False)
display(revenue_corr)

## 6. Clasificación de Películas por Rating

### 6.1 Creación de Categorías

In [ ]:
def classify_rating(rating):
    if rating >= 8.0:
        return 'Excelente'
    elif rating >= 7.0:
        return 'Buena'
    elif rating >= 6.0:
        return 'Regular'
    else:
        return 'Mala'

imdb_data['Class'] = imdb_data['Rating'].apply(classify_rating)

print("Distribución de Clases:")
display(imdb_data['Class'].value_counts())

plt.figure(figsize=(8, 5))
imdb_data['Class'].value_counts().plot(kind='bar')
plt.title('Distribución de Películas por Clase')
plt.xlabel('Clase')
plt.ylabel('Cantidad')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)
plt.show()

### 6.2 Análisis de Ingresos por Clase

In [ ]:
class_revenue = imdb_data.groupby('Class')['Revenue (Millions)'].agg(['mean', 'median', 'min', 'max'])
print("Estadísticas de Ingresos por Clase:")
display(class_revenue)

plt.figure(figsize=(10, 5))
imdb_data.boxplot(column='Revenue (Millions)', by='Class')
plt.title('Ingresos por Clase de Película')
plt.suptitle('')
plt.xlabel('Clase')
plt.ylabel('Revenue (Millions USD)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
genre_rating = imdb_data.groupby('Genre')['Rating'].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 5))
genre_rating.plot(kind='bar')
plt.title('Top 10 Géneros con Mejor Rating Promedio')
plt.xlabel('Género')
plt.ylabel('Rating Promedio')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Géneros con mejor rating promedio:")
display(genre_rating)

## 7. Limpieza de Datos

### 7.1 Eliminación de Datos Faltantes

In [ ]:
# Crear una copia limpia
imdb_clean = imdb_data.copy()

# Eliminar filas con datos faltantes en columnas clave
imdb_clean = imdb_clean.dropna(subset=['Revenue (Millions)', 'Metascore'])

print(f"Datos originales: {len(imdb_data)} filas")
print(f"Datos después de limpieza: {len(imdb_clean)} filas")
print(f"Filas eliminadas: {len(imdb_data) - len(imdb_clean)}")

In [ ]:
# Métodos de ordenamiento en Pandas

# 1. sort_values() - Ordenar por una columna
print("1. Ordenar por Rating (descendente):")
display(imdb_data[['Title', 'Rating', 'Revenue (Millions)']].sort_values('Rating', ascending=False).head(5))

# 2. sort_values() con múltiples columnas
print("\n2. Ordenar por Rating (desc) y Revenue (desc):")
display(imdb_data[['Title', 'Rating', 'Revenue (Millions)']].sort_values(['Rating', 'Revenue (Millions)'], ascending=[False, False]).head(5))

# 3. sort_index() - Ordenar por índice
print("\n3. Ordenar por índice (descendente):")
display(imdb_data[['Title', 'Rating']].sort_index(ascending=False).head(5))

In [ ]:
# Agregar columna de año de diferencia
# Como no tenemos una segunda obra, usaremos el año de estreno vs año actual
current_year = 2024
imdb_data['Years_Since_Release'] = current_year - imdb_data['Year']

print("Películas más antiguas (mayor tiempo desde estreno):")
display(imdb_data[['Title', 'Year', 'Years_Since_Release']].sort_values('Years_Since_Release', ascending=False).head(10))

print("\nEstadísticas de antigüedad de películas:")
print(f"  Promedio: {imdb_data['Years_Since_Release'].mean():.1f} años")
print(f"  Mínimo: {imdb_data['Years_Since_Release'].min():.0f} años")
print(f"  Máximo: {imdb_data['Years_Since_Release'].max():.0f} años")

In [ ]:
# Nuevos escritores (autores/directores)
new_writers = pd.DataFrame({
    'Director': ['Denis Villeneuve', 'Greta Gerwig', 'Christopher Nolan', 'Ava DuVernay', 'Bong Joon-ho'],
    'Obra_Principal': ['Dune: Part Two', 'Barbie', 'Oppenheimer', 'Origin', 'Parasite'],
    'Año': [2024, 2023, 2023, 2024, 2019],
    'Género': ['Sci-Fi', 'Comedy', 'Biography', 'Drama', 'Thriller'],
    'Rating': [8.8, 7.2, 8.6, 6.8, 8.5],
    'Revenue_Millions': [700, 600, 950, 20, 260]
})

print("Nuevos directores agregados:")
display(new_writers)

# Crear tabla de obras adicionales (segunda obra)
new_ouvres = pd.DataFrame({
    'Director': ['Denis Villeneuve', 'Greta Gerwig', 'Christopher Nolan', 'Ava DuVernay', 'Bong Joon-ho'],
    'Obra_2': ['Arrival', 'Little Women', 'The Dark Knight', 'Selma', 'Memories of Murder'],
    'Año_2': [2016, 2019, 2008, 2014, 2003],
    'Género_2': ['Sci-Fi', 'Drama', 'Action', 'History', 'Crime'],
    'Rating_2': [8.0, 7.8, 9.0, 7.5, 8.1]
})

print("\nSegundas obras:")
display(new_ouvres)

# Combinar con LEFT JOIN
writers_combined = pd.merge(new_writers, new_ouvres, on='Director', how='left')
writers_combined['Diferencia_Años'] = writers_combined['Año'] - writers_combined['Año_2']

print("\nTabla combinada con LEFT JOIN:")
display(writers_combined)

print("\nEstadísticas de diferencia de años:")
print(f"  Promedio: {writers_combined['Diferencia_Años'].mean():.1f} años")
print(f"  Mínimo: {writers_combined['Diferencia_Años'].min():.0f} años")
print(f"  Máximo: {writers_combined['Diferencia_Años'].max():.0f} años")

## 8. Conclusiones del Análisis

### Resumen de Hallazgos

1. **Datos Generales:**
   - El dataset contiene 1000 películas con 12 variables
   - Las columnas con datos faltantes son: Revenue (Millions) con 128 valores NaN y Metascore con 64 valores NaN
   - La mayoría de las películas son del período 2010-2016

2. **Calificaciones (Rating):**
   - El rating promedio es 6.72, con una mediana de 6.8
   - La distribución es aproximadamente normal con sesgo hacia ratings altos
   - La mayoría de las películas tienen ratings entre 6.0 y 7.4

3. **Ingresos (Revenue):**
   - El ingreso promedio es $82.96M, pero la mediana es $47.99M, indicando una distribución sesgada a la derecha
   - Hay películas con ingresos extremadamente altos (hasta $936M) que son outliers
   - La correlación entre rating y revenue es positiva pero débil (0.21)

4. **Géneros:**
   - Los géneros más comunes son Drama, Comedy, Action, Adventure y Thriller
   - Los géneros con mejor rating promedio son Drama, Biography y History

5. **Directores:**
   - Los directores con más películas son Christopher Nolan, Steven Spielberg, Ridley Scott y David Ayer
   - Christopher Nolan tiene el rating promedio más alto entre los directores con múltiples películas

6. **Relaciones y Correlaciones:**
   - Rating y Metascore tienen correlación positiva fuerte (0.65)
   - Votes y Revenue tienen correlación positiva moderada (0.61)
   - Runtime tiene correlación débil con otras variables

### Recomendaciones

1. **Para análisis más profundos:**
   - Investigar los outliers con alto rating pero bajo revenue
   - Analizar la evolución de ratings y revenues a lo largo de los años

2. **Para limpieza de datos:**
   - Considerar imputación de datos faltantes en Revenue y Metascore
   - Analizar la distribución de géneros para identificar posibles sesgos

3. **Para futuros análisis:**
   - Explorar relaciones entre el elenco de actores y el éxito de las películas
   - Analizar la evolución de ratings por género a lo largo del tiempo
   - Comparar la crítica (Metascore) vs la audiencia (Rating)

In [ ]:
print("\n" + "="*60)
print("RESUMEN FINAL DEL ANÁLISIS EXPLORATORIO")
print("="*60)
print(f"\n📊 Dataset: {len(imdb_data)} películas, {len(imdb_data.columns)} variables")
print(f"📈 Rating promedio: {imdb_data['Rating'].mean():.2f}")
print(f"💰 Revenue promedio: ${imdb_data['Revenue (Millions)'].mean():.2f}M")
print(f"🎬 Género más común: {genres.value_counts().index[0]}")
print(f"👨‍🎨 Director con más películas: {director_counts.index[0]} ({director_counts.iloc[0]} películas)")
print(f"📅 Año con más películas: {year_counts.idxmax()}")
print(f"📊 Correlación Rating-Revenue: {correlation:.3f}")
print("="*60)